# NeuralOps Notebook 04 — Benchmarking, torch.compile, and ONNX export

In [1]:

import tempfile
from pathlib import Path
import torch
import torch.nn as nn
import neuralops

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class TinyLM(nn.Module):
    def __init__(self, vocab=32, hidden=16):
        super().__init__()
        self.config = type("Config", (), {"vocab_size": vocab})()
        self.emb = nn.Embedding(vocab, hidden)
        self.proj = nn.Linear(hidden, vocab)
    def forward(self, input_ids):
        return self.proj(self.emb(input_ids))
model = TinyLM().to(device).eval()
tokens = torch.randint(0, 32, (2, 8), device=device)
print("device:", device)


device: cuda


In [2]:

table = neuralops.quick_benchmark(model, task="synthetic_sst2", device=str(device), iterations=3)
assert float(table.iloc[0]["latency_ms"]) > 0
assert bool(table.iloc[0]["finite"])


/mnt/d/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


          task device  latency_ms  finite
synthetic_sst2   cuda    0.036012    True


In [3]:

compiled = torch.compile(model, fullgraph=True, backend="eager")
with torch.no_grad():
    compiled_output = compiled(tokens)
assert torch.isfinite(compiled_output).all()
print("torch.compile output:", tuple(compiled_output.shape), "finite:", bool(torch.isfinite(compiled_output).all()))


torch.compile output: (2, 8, 32) finite: True


In [4]:

import onnx
export_dir = Path(tempfile.mkdtemp(prefix="neuralops_notebook_onnx_"))
onnx_path = neuralops.export_model(
    model, export_dir, format="onnx", sample_input=tokens.detach().cpu(), opset_version=17
)
onnx.checker.check_model(str(onnx_path))
print("validated ONNX:", onnx_path)


2026-10-19 19:49:11,481 [INFO] Finalized surgery across 0 UniversalAttentionBridge modules.


W1019 19:49:11.552000 132620 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


2026-10-19 19:49:12,102 [INFO] No unused functions to remove


2026-10-19 19:49:12,103 [WARNING] The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


2026-10-19 19:49:12,108 [INFO] No unused functions to remove


...

2026-10-19 19:49:12,112 [INFO] Skipping constant folding for node 'node_embedding' because it is graph input to preserve graph signature


2026-10-19 19:49:12,113 [INFO] No unused functions to remove


2026-10-19 19:49:12,113 [INFO] PassManager: No more graph changes detected after step 1


2026-10-19 19:49:12,114 [INFO] Exported model to ONNX: /tmp/neuralops_notebook_onnx_nykpojz0/model.onnx


[torch.onnx] Obtain model graph for `_TensorOnlyExportWrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `_TensorOnlyExportWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
validated ONNX: /tmp/neuralops_notebook_onnx_nykpojz0/model.onnx
